# E4: Notebook (temporary only)

In [1]:
import hashlib
import json
from datetime import datetime, timezone
from pathlib import Path
import pandas as pd
import sys

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.common.config import load_config

cfg = load_config()
proto = cfg["protocol"]
SNAPSHOT_ID = cfg["snapshot_id"]
PROTOCOL_ID = proto["id"]
TIMEZONE = cfg["timezone"]

DATA_DIR = ROOT / cfg["paths"]["processed"] / str(SNAPSHOT_ID)
EVAL_DIR = ROOT / cfg["paths"]["evaluation"] / str(PROTOCOL_ID)
MANIFEST_PATH = ROOT / cfg["paths"]["manifests"] / f"{SNAPSHOT_ID}.json"
OLD_QUALITY_PATH = ROOT / "data" / "processed" / "flux-20261115" / "inventory_quality.csv"

# protocol config
PUBLICATION_LAG_HOURS = proto["publication_lag_hours"]
BUDGETS = proto["local_budget_hours"]
PERIODS = {
    "validation": (str(proto["train_until"]), str(proto["validation_until"])),
    "test": (str(proto["test_start"]), str(proto["test_end"])),
}
ROLES = {
    "S_train": ("known_site", ["validation", "test"]),
    "S_dev_new": ("new_site", ["validation"]),
    "S_test_new": ("new_site", ["test"]),
}
CASE_COLUMNS = [
    "case_id", "counter_id", "site_group_id", "origin", "scenario",
    "temporal_split", "spatial_role", "local_budget_hours", "support_start",
    "support_end", "protocol_id", "snapshot_id",
]

# E1 Files
counts = pd.read_csv(DATA_DIR / "counts.csv", dtype={"counter_id": str, "is_observed": str, "is_scoreable": str})
counters = pd.read_csv(DATA_DIR / "counters.csv", dtype=str)
quality = pd.read_csv(DATA_DIR / "inventory_quality.csv", dtype=str)
context = pd.read_csv(DATA_DIR / "context.csv", dtype=str)
with open(DATA_DIR / "selection_summary.json", encoding="utf-8") as f:
    summary = json.load(f)
site_groups = summary["site_groups"]

def inspect_df(df, name, limit=20):
    print(f"=== {name} ({len(df):,} rows x {len(df.columns)} cols) ===")
    for col in df.columns:
        vals = df[col].dropna().unique()
        print(f"  {col} ({len(vals)} unique): {vals[:limit]}")
    print()

def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def to_text(ts):
    return ts.strftime("%Y-%m-%dT%H:%M:%S+00:00")

## EDA and checks of E1 files

In [2]:
for name, df in [("context", context), ("counts", counts), ("counters", counters), ("quality", quality)]:
    inspect_df(df, name)

print("period in counts.csv:", counts["timestamp"].min(), "->", counts["timestamp"].max())
print("counters:", counts["counter_id"].nunique(), "| hours per counter:", counts.groupby("counter_id").size().unique())
print("site groups per list:", {k: len(v) for k, v in site_groups.items()})
print("status of counters:", summary["counters_by_status"])

=== context (165,405 rows x 12 cols) ===
  context_source_id (5 unique): ['jours_feries' 'vacances_zone_c' 'meteo_prevue' 'meteo_observee'
 'greves_idf']
  location_id (6 unique): ['FR-metropole' 'zone-C' 'openmeteo-jma_gsm-48.8566-2.3522'
 'openmeteo-48.8566-2.3522' 'meteofrance-75114001' 'IDF-transports']
  valid_time (27025 unique): ['2023-10-16T00:00:00+00:00' '2023-10-16T01:00:00+00:00'
 '2023-10-16T02:00:00+00:00' '2023-10-16T03:00:00+00:00'
 '2023-10-16T04:00:00+00:00' '2023-10-16T05:00:00+00:00'
 '2023-10-16T06:00:00+00:00' '2023-10-16T07:00:00+00:00'
 '2023-10-16T08:00:00+00:00' '2023-10-16T09:00:00+00:00'
 '2023-10-16T10:00:00+00:00' '2023-10-16T11:00:00+00:00'
 '2023-10-16T12:00:00+00:00' '2023-10-16T13:00:00+00:00'
 '2023-10-16T14:00:00+00:00' '2023-10-16T15:00:00+00:00'
 '2023-10-16T16:00:00+00:00' '2023-10-16T17:00:00+00:00'
 '2023-10-16T18:00:00+00:00' '2023-10-16T19:00:00+00:00']
  local_date (1127 unique): ['2023-10-16' '2023-10-17' '2023-10-18' '2023-10-19' '2023-10-2

In [3]:
# manifest SHA-256 check
manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
for f in manifest["files"]:
    p = ROOT / f["path"]
    if f["path"].startswith(cfg["paths"]["processed"]) and p.exists():
        same = sha256(p) == f["sha256"]
        print("OK " if same else "!! DIFFERENT", f["path"])

# we score only is_scoreable = true, and zeros must not be replaced automatically
observed = counts["is_observed"].str.lower() == "true"
scoreable = counts["is_scoreable"].str.lower() == "true"
count_raw_num = pd.to_numeric(counts["count_raw"], errors="coerce")
count_num = pd.to_numeric(counts["count"], errors="coerce")
measured_zero = observed & (count_raw_num == 0)

print("\ncounts.csv checks")
print("duplicated (counter_id, timestamp):", counts.duplicated(["counter_id", "timestamp"]).sum())
print("scoreable but not observed:", (scoreable & ~observed).sum())
print("scoreable with count != count_raw:", (scoreable & (count_num != count_raw_num)).sum())
print("filled values in `count` (not scoreable):", (~scoreable & count_num.notna()).sum())
print("measured zeros:", measured_zero.sum(), "| NOT scoreable:", (measured_zero & ~scoreable).sum())

# whole sites not in training
train, dev, test = set(site_groups["train"]), set(site_groups["dev_new"]), set(site_groups["test_new"])
roles = quality[quality["period_role"] == "selection"][["counter_id", "site_role"]].merge(
    counters[["counter_id", "site_group_id"]], on="counter_id"
)
several = roles[roles["site_role"] != "none"].groupby("site_group_id")["site_role"].nunique()
print("\nsite_groups checks")
print("groups in two lists:", (train & dev) | (train & test) | (dev & test))
print("train counters inside a reserved group:", ((roles["site_role"] == "S_train") & roles["site_group_id"].isin(dev | test)).sum())
print("groups with several roles:", (several > 1).sum())

# hours with no data anywhere must be counted
share_observed = observed.groupby(counts["timestamp"]).mean()
empty_hours = share_observed[share_observed < 0.1]
print("\nhours observed by <10% of counters:", len(empty_hours))
print(pd.Series(pd.to_datetime(empty_hours.index, utc=True).tz_convert(TIMEZONE).date).value_counts().sort_index())

# comparison with previous version
if OLD_QUALITY_PATH and OLD_QUALITY_PATH.exists():
    old = pd.read_csv(OLD_QUALITY_PATH, dtype=str).query("period_role == 'selection'").set_index("counter_id")["site_role"]
    new = quality.query("period_role == 'selection'").set_index("counter_id")["site_role"]
    changed = [(c, old[c], new[c]) for c in old.index.intersection(new.index) if old[c] != new[c]]
    print("\ncounters whose role changed:", len(changed))
    for c in changed:
        print("   ", c)

OK  data/processed/hist-20261117/counts.csv
OK  data/processed/hist-20261117/counters.csv
OK  data/processed/hist-20261117/audit_duplicates.csv
OK  data/processed/hist-20261117/inventory_quality.csv
OK  data/processed/hist-20261117/selection_summary.json
OK  data/processed/hist-20261117/context.csv
OK  data/processed/hist-20261117/context_sources.json
OK  data/processed/hist-20261117/context_join_audit.json
OK  data/processed/hist-20261117/fill_report.json

counts.csv checks
duplicated (counter_id, timestamp): 0
scoreable but not observed: 0
scoreable with count != count_raw: 0
filled values in `count` (not scoreable): 36350
measured zeros: 110896 | NOT scoreable: 17965

site_groups checks
groups in two lists: set()
train counters inside a reserved group: 0
groups with several roles: 0

hours observed by <10% of counters: 104
2023-10-29     1
2024-10-27     1
2025-10-26     1
2026-07-02     2
2026-07-06     2
2026-08-22    24
2026-09-23    24
2026-10-04    24
2026-10-25    25
Name: cou

## Building cases.csv

In [4]:
assert SNAPSHOT_ID, "snapshot_id is empty in config/project.yaml"

def build_cases(selected, counters):
    rows = []
    for _, c in selected.iterrows():
        scenario, splits = ROLES[c["site_role"]]
        for split in splits:
            origins = pd.date_range(*PERIODS[split], freq="D").tz_localize(TIMEZONE).tz_convert("UTC")
            budgets = BUDGETS if scenario == "new_site" else [None]
            for origin in origins:
                for budget in budgets:
                    if budget is None:
                        s_start, s_end = "", ""
                    else:
                        end = origin - pd.Timedelta(hours=PUBLICATION_LAG_HOURS)
                        s_start, s_end = to_text(end - pd.Timedelta(hours=budget)), to_text(end)
                    b_tag = "" if budget is None else f"_b{budget}"
                    rows.append({
                        "case_id": f"{c['counter_id']}_{origin:%Y%m%dT%H}_{scenario}{b_tag}",
                        "counter_id": c["counter_id"],
                        "origin": to_text(origin),
                        "scenario": scenario,
                        "temporal_split": split,
                        "spatial_role": c["site_role"],
                        "local_budget_hours": "" if budget is None else budget,
                        "support_start": s_start,
                        "support_end": s_end,
                        "protocol_id": PROTOCOL_ID,
                        "snapshot_id": SNAPSHOT_ID,
                    })
    return pd.DataFrame(rows).merge(counters[["counter_id", "site_group_id"]], on="counter_id", how="left")[CASE_COLUMNS]

def check_cases(cases, e1_groups, last_data):
    reserved = set(e1_groups["dev_new"]) | set(e1_groups["test_new"])
    train = cases[cases["spatial_role"] == "S_train"]
    last_target = pd.to_datetime(cases["origin"], utc=True) + pd.Timedelta(hours=23)
    problems = {
        "duplicate case_id": cases["case_id"].duplicated().sum(),
        "empty site_group_id": cases["site_group_id"].isna().sum(),
        "site groups with multiple roles": (cases.groupby("site_group_id")["spatial_role"].nunique() > 1).sum(),
        "S_train cases in a reserved group": train["site_group_id"].isin(reserved).sum(),
        "cases whose 24h exceed available data": (last_target > last_data).sum(),
    }
    for name, value in problems.items():
        print(("OK  " if value == 0 else "!!  ") + f"{name}: {int(value)}")
    return problems

selected = quality[(quality["period_role"] == "selection") & (quality["selection_status"] != "excluded")]
last_data = pd.to_datetime(counts["timestamp"], utc=True).max()

print(f"{PROTOCOL_ID} (statut : {proto['status']}) sur {SNAPSHOT_ID}")
cases = build_cases(selected, counters)
print(f"{len(selected)} compteurs -> {len(cases)} cas\n")
problems = check_cases(cases, site_groups, last_data)
assert not any(problems.values()), "A check failed, cases.csv is not generated."

print("\n", cases.groupby(["scenario", "temporal_split", "local_budget_hours"], dropna=False).size().to_string())

EVAL_DIR.mkdir(parents=True, exist_ok=True)
cases.to_csv(EVAL_DIR / "cases.csv", index=False)

inputs = ["inventory_quality.csv", "counters.csv", "selection_summary.json", "counts.csv"]
manifest_out = {
    "generated_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "script": "src/eval/make_cases.py",
    "snapshot_id": SNAPSHOT_ID,
    "protocol_id": PROTOCOL_ID,
    "periods": PERIODS,
    "publication_lag_hours": PUBLICATION_LAG_HOURS,
    "budgets": BUDGETS,
    "n_cases": len(cases),
    "last_data_timestamp": str(last_data),
    "inputs_sha256": {n: sha256(DATA_DIR / n) for n in inputs},
    "cases_sha256": sha256(EVAL_DIR / "cases.csv"),
}
(EVAL_DIR / "cases_manifest.json").write_text(json.dumps(manifest_out, indent=2, ensure_ascii=False), encoding="utf-8")
print("\nout :", EVAL_DIR / "cases.csv")

proto-v1 (statut : proposed) sur hist-20261117
112 compteurs -> 12659 cas

OK  case_id en double: 0
OK  site_group_id vide: 0
OK  groupes à plusieurs rôles: 0
OK  cas S_train dans un groupe réservé: 0
OK  cas dont les 24 h dépassent les données: 0

 scenario    temporal_split  local_budget_hours
known_site  test                                  4365
            validation                            6014
new_site    test            24                     450
                            72                     450
                            168                    450
            validation      24                     310
                            72                     310
                            168                    310

écrit : /Users/sherlyn/Documents/ISD Master/ParisPulse/data/evaluation/proto-v1/cases.csv


## CHECKS ON FORECASTS FILES (E2,E3)

In [5]:
META_PATH = None
TEMPLATE_PATH = ROOT / "doc" / "templates" / "forecasts.csv"
PRED_FILES = [
    ROOT / "data" / "prediction_results" / "EXP-001" / "e2-reference_profile-measured.pred.csv",
    ROOT / "data" / "prediction_results" / "EXP-001" / "e2-reference-measured.pred.csv",
]

pred = pd.concat([pd.read_csv(p, dtype={"counter_id": str}, low_memory=False) for p in PRED_FILES], ignore_index=True)
cases_df = pd.read_csv(EVAL_DIR / "cases.csv", dtype=str)

template_cols = list(pd.read_csv(TEMPLATE_PATH).columns)
print("Missing columns:", [c for c in template_cols if c not in pred.columns])
print("Extra columns:", [c for c in pred.columns if c not in template_cols])
print("All-empty columns:", [c for c in pred.columns if pred[c].isna().all()])

for col in ["origin", "target_timestamp", "fit_cutoff"]:
    pred[col] = pd.to_datetime(pred[col], utc=True)
cases_df["origin"] = pd.to_datetime(cases_df["origin"], utc=True)
counts_ts = pd.to_datetime(counts["timestamp"], utc=True)
pred["y_pred"] = pd.to_numeric(pred["y_pred"], errors="coerce")

# duplicates, complete 24h cases and failure causes
print("\nDuplicates (case_id, run, horizon):", pred.duplicated(["case_id", "method_run_id", "horizon"]).sum())
expected_cases = set(cases_df["case_id"])
valid_hours = pred.assign(has_val=pred["y_pred"].notna()).groupby(["method_run_id", "case_id"])["has_val"].sum().reset_index(name="vh")

summary_df = pd.DataFrame([
    {
        "run": run,
        "expected": len(expected_cases),
        "no row at all": len(expected_cases - set(g["case_id"])),
        "all empty": int((g["vh"] == 0).sum()),
        "partial (1-23 h)": int(((g["vh"] > 0) & (g["vh"] < 24)).sum()),
        "complete (24 h)": int((g["vh"] == 24).sum()),
    }
    for run, g in valid_hours.groupby("method_run_id")
])
print("\n", summary_df.to_string(index=False))
print("Cases delivered but not in cases.csv:", len(set(pred["case_id"]) - expected_cases))
print("Negative y_pred:", (pred["y_pred"] < 0).sum())
print("prediction_status:", pred["prediction_status"].value_counts(dropna=False).to_dict())
print("method_used:", pred["method_used"].value_counts(dropna=False).to_dict())
print("Failure reasons:\n", pred["status_reason"].value_counts(dropna=False).head(4).to_dict())

# shifted timestamps and origins
merged = pred.merge(cases_df[["case_id", "origin"]], on="case_id", how="left", suffixes=("", "_expected"))
print("\nOrigin different from cases.csv:", ((merged["origin"] != merged["origin_expected"]) & merged["origin_expected"].notna()).sum())
expected_ts = pred["origin"] + pd.to_timedelta(pred["horizon"] - 1, unit="h")
print("target_timestamp != origin + (horizon - 1)h:", (pred["target_timestamp"] != expected_ts).sum())

valid_pairs = pd.DataFrame({"counter_id": counts["counter_id"], "timestamp": counts_ts, "exists": True})
joined = pred.merge(valid_pairs, left_on=["counter_id", "target_timestamp"], right_on=["counter_id", "timestamp"], how="left")
print("Predicted hours not found in counts.csv:", joined["exists"].isna().sum())

# data splits, leakage
test_cases = cases_df[cases_df["temporal_split"] == "test"]
test_rows = pred[pred["case_id"].isin(test_cases["case_id"])]
print("\nEmpty fit_cutoff:", pred["fit_cutoff"].isna().sum())
print("fit_cutoff after origin:", (pred["fit_cutoff"] > pred["origin"]).sum())
print("Test forecasts fitted after test start:", (test_rows["fit_cutoff"] > test_cases["origin"].min()).sum())

if META_PATH is not None:
    meta = json.load(open(META_PATH, encoding="utf-8"))
    print("Reserved sites seen in training:", set(meta["training_site_groups"]) & (dev | test))

# all models must use the same input data for new sites
for col in ["snapshot_id", "protocol_id"]:
    if col in pred.columns:
        print(f"{col} different from cases.csv:", (~pred[col].astype(str).isin(cases_df[col])).sum())

new_sites = pred[pred["scenario"] == "new_site"]
if len(new_sites):
    print("New-site cases with several local supports:", (new_sites.groupby("case_id")["support_id"].nunique() > 1).sum())

# coverage of cases.csv
done = set(pred["case_id"])
print("\nCoverage of cases.csv:")
for (scenario, split), g in cases_df.groupby(["scenario", "temporal_split"]):
    ids = set(g["case_id"])
    print(f"    {scenario}/{split} : {len(ids & done)} / {len(ids)}")

Missing columns: []
Extra columns: []
All-empty columns: ['model_run_id', 'fit_cutoff', 'support_id', 'adaptation_id', 'context_version', 'feature_version']

Duplicates (case_id, run, horizon): 0

                           run  expected  no row at all  all empty  partial (1-23 h)  complete (24 h)
        e2-reference-measured     12659           7141          0                 0            10858
e2-reference_profile-measured     12659           7141          0                 0            10858
Cases delivered but not in cases.csv: 5340
Negative y_pred: 0
prediction_status: {'ok': 496064, 'fallback': 25120}
method_used: {'weekly_profile_median': 251596, 'weekly_naive': 244468, 'profile_median_dow_hour': 25120}
Failure reasons:
 {nan: 496064, 'lag168_missing': 16124, 'profile_weeks_missing': 8996}

Origin different from cases.csv: 0
target_timestamp != origin + (horizon - 1)h: 0
Predicted hours not found in counts.csv: 0

Empty fit_cutoff: 521184
fit_cutoff after origin: 0
Test forecas

## Findings/conclusion

In [6]:
CASE_COLUMNS = [  
    "case_id", "counter_id", "site_group_id", "origin", "scenario",
    "temporal_split", "spatial_role", "local_budget_hours", "support_start",
    "support_end", "protocol_id", "snapshot_id",
]
